# 6-4 zero_grad, backward, step 순서 — 기본

직접 작성한 코드와 저장된 실행 결과를 정리했습니다.


In [1]:
import random
import json
import math
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset, random_split

# 실습 결과가 매번 비슷하게 나오도록 seed를 고정합니다.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


In [2]:
w = torch.tensor([1.0], requires_grad=True)
loss1 = (w * 2 - 5).pow(2).mean()
loss1.backward()
# TODO: 첫 번째 grad를 복사하세요.
grad_after_first = w.grad.clone()

loss2 = (w * 3 - 4).pow(2).mean()
# TODO: 두 번째 backward 전에 gradient를 초기화하세요.
w.grad.zero_()
loss2.backward()

grad_after_second = w.grad.clone()
print('first grad:', grad_after_first)
print('second grad:', grad_after_second)

first grad: tensor([-12.])
second grad: tensor([-6.])


In [5]:
def batch_step(model, batch, loss_fn, optimizer):
    # TODO: batch에서 x, y를 꺼내세요.
    x, y = batch
    pred = model(x)
    # TODO: loss를 계산하세요.
    loss = loss_fn(pred, y)
    # TODO: 업데이트 순서를 완성하세요.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.detach())

x = torch.randn(16, 3)
y = torch.randn(16, 1)
model = nn.Linear(3, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
print(batch_step(model, (x, y), nn.MSELoss(), optimizer))

1.7955540418624878


In [6]:
def run_step(model, x, y, optimizer, use_zero_grad):
    pred = model(x)
    loss = nn.MSELoss()(pred, y)
    # TODO: use_zero_grad가 True일 때만 gradient를 초기화하세요.
    if use_zero_grad:
        optimizer.zero_grad()
    loss.backward()
    norm = sum(float(p.grad.norm()) for p in model.parameters() if p.grad is not None)
    optimizer.step()
    return norm

torch.manual_seed(0)
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(4, 2)
y = torch.randn(4, 1)
print('norm1:', run_step(model, x, y, optimizer, True))
print('norm2:', run_step(model, x, y, optimizer, False))

norm1: 2.679388165473938
norm2: 5.281521439552307
